# IDE Configuration for MaaS on OpenShift

This notebook configures popular IDEs to use models served via MaaS (Models as a Service) on RHOAI.

| Aspect | Direct to RHOAI | Via MaaS |
|--------|----------------|----------|
| Endpoint | InferenceService Route URL per model | MaaS Gateway URL |
| Auth | SA token or OAuth per model | MaaS API key (`sk-oai-*`) |
| Rate Limiting | None | Subscription-based limits |
| Model Discovery | Manual | `GET /v1/models` |
| Key Management | Manual token rotation | Create/revoke via Dashboard or API |

## 1. Get MaaS Endpoint and Models

In [ ]:
import subprocess
import json
import urllib.request
import ssl

result = subprocess.run(
    ["kubectl", "get", "ingresses.config.openshift.io", "cluster",
     "-o", "jsonpath={.spec.domain}"],
    capture_output=True, text=True
)
CLUSTER_DOMAIN = result.stdout.strip()
MAAS_HOST = f"https://maas.{CLUSTER_DOMAIN}"

token_result = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
OC_TOKEN = token_result.stdout.strip()

print(f"MaaS Gateway: {MAAS_HOST}")
print("")

# List models
ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

req = urllib.request.Request(
    f"{MAAS_HOST}/maas-api/v1/models",
    headers={"Authorization": f"Bearer {OC_TOKEN}", "Content-Type": "application/json"}
)
try:
    with urllib.request.urlopen(req, context=ctx) as resp:
        models_data = json.loads(resp.read())
    print("Available Models:")
    for model in models_data.get("data", []):
        print(f"  Model: {model['id']}")
        print(f"  URL:   {model.get('url', 'N/A')}")
        print("")
except Exception as e:
    print(f"⚠️  Could not list models: {e}")
    print("   Ensure MaaS is enabled and models are deployed.")

## 2. Get an API Key

### Option A: Through the RHOAI Dashboard (Recommended)

1. Navigate to **AI assets → Endpoints → Models as a Service**
2. Click **View** on your model
3. Click **Generate Token**
4. Copy and save the key — it is shown only once

> **Note:** The URL is displayed as "http" in the Dashboard, but **"https" access works**.

### Option B: Through CLI

In [ ]:
%%bash
CLUSTER_DOMAIN=$(kubectl get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
HOST="https://maas.${CLUSTER_DOMAIN}"

echo "Creating API key for IDE access..."
API_KEY_RESP=$(curl -sSk -X POST "${HOST}/maas-api/v1/api-keys" \
  -H "Authorization: Bearer $(oc whoami -t)" \
  -H "Content-Type: application/json" \
  -d '{"name": "ide-key", "description": "IDE access", "expiresIn": "30d"}')

echo $API_KEY_RESP | python3 -m json.tool

API_KEY=$(echo $API_KEY_RESP | python3 -c "import sys,json; print(json.load(sys.stdin).get('key',''))")
echo ""
echo "⚠️  Save this API key — it is shown only once!"
echo "   Use it as 'API Key' in your IDE configuration below."

## 3. IDE Configuration

Run the cell below to generate config snippets for each IDE using your actual model URLs.

**Replace `sk-oai-YOUR-KEY` with the API key from step 2.**

In [ ]:
# Gather model info
try:
    model_name = models_data["data"][0]["id"]
    model_url = models_data["data"][0]["url"]
except (NameError, IndexError, KeyError):
    model_name = "MODEL_NAME"
    model_url = "MODEL_URL"
    print("⚠️  Could not detect model info — using placeholders.")
    print("   Run cell 1 first, or replace MODEL_NAME and MODEL_URL manually.")
    print("")

API_KEY_PLACEHOLDER = "sk-oai-YOUR-KEY"

print(f"Model Name: {model_name}")
print(f"Model URL:  {model_url}")
print(f"API Key:    {API_KEY_PLACEHOLDER}  ← replace with your key")

### Cursor

Settings → Models → OpenAI API Key

In [ ]:
print("=== Cursor Settings ===")
print(f"  Base URL: {model_url}/v1")
print(f"  API Key:  {API_KEY_PLACEHOLDER}")
print(f"  Model:    {model_name}")
print("")

# MCP tools config
routes_result = subprocess.run(
    ["oc", "get", "routes", "-n", "mcp-servers",
     "-o", "jsonpath={range .items[*]}{.metadata.name}={.spec.host}\n{end}"],
    capture_output=True, text=True
)
mcp_urls = {"context7": "https://mcp.context7.com/mcp"}
for line in routes_result.stdout.strip().split("\n"):
    if "=" in line:
        name, host = line.split("=", 1)
        mcp_urls[name.replace("mcp-", "")] = f"https://{host}/sse"

cursor_config = {"mcpServers": {n: {"url": u} for n, u in mcp_urls.items()}}
print("=== .cursor/mcp.json (MCP tools) ===")
print(json.dumps(cursor_config, indent=2))

### VS Code (Continue.dev Extension)

In [ ]:
continue_config = {
    "models": [
        {
            "title": f"{model_name} (RHOAI via MaaS)",
            "provider": "openai",
            "model": model_name,
            "apiBase": f"{model_url}/v1",
            "apiKey": API_KEY_PLACEHOLDER
        }
    ],
    "tabAutocompleteModel": {
        "title": "Autocomplete (RHOAI)",
        "provider": "openai",
        "model": model_name,
        "apiBase": f"{model_url}/v1",
        "apiKey": API_KEY_PLACEHOLDER
    }
}

print("=== ~/.continue/config.json ===")
print(json.dumps(continue_config, indent=2))

### Claude Code

In [ ]:
print("=== Claude Code environment variables ===")
print(f'export OPENAI_BASE_URL="{model_url}/v1"')
print(f'export OPENAI_API_KEY="{API_KEY_PLACEHOLDER}"')

### OpenCode

In [ ]:
opencode_config = {
    "provider": {
        "rhoai": {
            "type": "openai",
            "baseURL": f"{model_url}/v1",
            "apiKey": API_KEY_PLACEHOLDER,
            "models": {
                model_name: {"name": model_name, "maxTokens": 8192}
            }
        }
    }
}

print("=== ~/.config/opencode/config.json ===")
print(json.dumps(opencode_config, indent=2))

## 4. Important: Direct Access vs MaaS Access

When MaaS is enabled, **two routes exist**:

| Route | Path | Auth Enforced? |
|-------|------|---------------|
| MaaS Gateway | `https://maas.apps.<domain>/maas-api/v1/...` | Yes (API key + rate limit) |
| Direct model | `https://maas.apps.<domain>/<namespace>/<model-id>/v1/...` | Only if "Require authentication" is checked |

**Always ensure "Require authentication" is enabled** alongside MaaS to prevent bypassing the gateway.

## 5. Verification

In [ ]:
%%bash
CLUSTER_DOMAIN=$(kubectl get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
HOST="https://maas.${CLUSTER_DOMAIN}"

echo "=== List Models via MaaS ==="
curl -sSk "${HOST}/maas-api/v1/models" \
  -H "Authorization: Bearer $(oc whoami -t)" | python3 -c "
import sys, json
data = json.load(sys.stdin)
for m in data.get('data', []):
    print(f'  ✅ {m[\"id\"]}  →  {m.get(\"url\", \"N/A\")}')
" 2>/dev/null || echo "  ⚠️  Could not list models"

In [ ]:
%%bash
CLUSTER_DOMAIN=$(kubectl get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
HOST="https://maas.${CLUSTER_DOMAIN}"

MODELS_JSON=$(curl -sSk "${HOST}/maas-api/v1/models" \
  -H "Authorization: Bearer $(oc whoami -t)" \
  -H "Content-Type: application/json")
MODEL_NAME=$(echo $MODELS_JSON | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data'][0]['id'] if d.get('data') else '')" 2>/dev/null)
MODEL_URL=$(echo $MODELS_JSON | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data'][0]['url'] if d.get('data') else '')" 2>/dev/null)

if [ -z "$MODEL_NAME" ]; then
    echo "⚠️  No models available for testing."
    exit 0
fi

echo "=== Quick Inference Test ==="
echo "Model: ${MODEL_NAME}"
echo ""

curl -sSk "${MODEL_URL}/v1/chat/completions" \
  -H "Authorization: Bearer $(oc whoami -t)" \
  -H "Content-Type: application/json" \
  -d "{\"model\": \"${MODEL_NAME}\", \"messages\": [{\"role\": \"user\", \"content\": \"Hello\"}], \"max_tokens\": 20}" | python3 -m json.tool

## Troubleshooting

| Issue | Solution |
|-------|----------|
| Connection refused | Check MaaS Gateway: `kubectl get gateway -n openshift-ingress maas-default-gateway` |
| 401 Unauthorized | Verify API key; generate a new one via Dashboard or `POST /maas-api/v1/api-keys` |
| 403 Forbidden | Check subscription access and model permissions |
| 429 Too Many Requests | Rate limit exceeded; wait or request higher subscription limits |
| Model not found | Verify model is deployed with MaaS enabled: check RHOAI Dashboard |
| 502 Bad Gateway | InferenceService not ready: `oc get inferenceservice -A` |
| SSL cert errors | URL may show as "http" in Dashboard; use "https" instead |